In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import LabelEncoder
from scipy.stats import rankdata

# 1. LOAD DATA
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_sub = pd.read_csv("sample_submission.csv")
test_ids = test['Id']

# 2. ADVANCED FEATURE ENGINEERING [4]
# Frequency encoding for School captures program prestige [5, 6]
school_freq = train['School'].value_counts().to_dict()

for df in [train, test]:
    # --- Body Composition & Athletic Context ---
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    df['Speed_Score'] = df['Weight'] / df['Sprint_40yd']
    df['Explosiveness'] = df['Vertical_Jump'] + df['Broad_Jump']
    df['Power_Index'] = df['Vertical_Jump'] * df['Weight']
    df['Total_Agility'] = df['Agility_3cone'] + df['Shuttle']
    df['Strength_Index'] = df['Bench_Press_Reps'] / df['Weight']
    
    # Position-Relative athleticism (Context is key for 0.85+) [4]
    df['Speed_pos_diff'] = df['Sprint_40yd'] - df.groupby('Position')['Sprint_40yd'].transform('mean')
    df['BMI_pos_diff'] = df['BMI'] - df.groupby('Position')['BMI'].transform('mean')
    
    # Missing indicators [4]
    for col in ['Agility_3cone', 'Shuttle', 'Bench_Press_Reps']:
        df[f'{col}_missing'] = df[col].isnull().astype(int)
    df['Total_Missing'] = df[['Agility_3cone_missing', 'Shuttle_missing', 'Bench_Press_Reps_missing']].sum(axis=1)

    # --- Reddit/Scientific Features ---
    # Catch Radius: Conglomerate of size, jump, and lateral quickness [7]
    df['Catch_Radius'] = df['Height'] + (df['Vertical_Jump']/100) + (df['Broad_Jump']/100) - (df['Shuttle']/10)
    
    # Peak Power (Sayers Formula) - Note: Use jump directly as it's already in cm [4]
    df['Peak_Power_Watts'] = (60.7 * df['Vertical_Jump']) + (45.3 * df['Weight']) - 2055
    
    # Age Bins: High ceiling for underclassmen [4]
    df['Is_Young_Prospect'] = (df['Age'] <= 21).astype(int)
    
    # School Signal
    df['School_Freq'] = df['School'].map(school_freq).fillna(1)

# 3. PREPROCESSING [6, 8]
train = train.drop(columns=["Id", "School"])
test = test.drop(columns=["Id", "School"])

# Label Encoding
le = LabelEncoder()
for col in ["Player_Type", "Position_Type", "Position"]:
    train[col] = le.fit_transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

X = train.drop(columns=["Drafted"])
y = train["Drafted"]

# MICE Imputer [8, 9]
imputer = IterativeImputer(random_state=2025)
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
test_imputed = pd.DataFrame(imputer.transform(test), columns=X.columns)

# 4. FIX NUMERICAL STABILITY (Avoids XGBoost Overflow Error) [3, 10]
for df in [X_imputed, test_imputed]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    # Simple fill for any remaining NaNs caused by inf division
    df.fillna(df.mean(), inplace=True)

# 5. TUNED MODELS [9, 11]
cat = CatBoostClassifier(iterations=250, depth=6, learning_rate=0.05, random_seed=2025, verbose=0)
xgb = XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.05, random_state=2025, eval_metric='logloss')
lgbm = LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=2025, objective='binary')

# 6. ENSEMBLE WITH RANK AVERAGING [1, 12]
# 10-fold CV provides higher stability for elite scores
rskf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=42)
test_cat = np.zeros(len(test_imputed))
test_xgb = np.zeros(len(test_imputed))
test_lgbm = np.zeros(len(test_imputed))

for fold, (train_idx, valid_idx) in enumerate(rskf.split(X_imputed, y)):
    X_t, y_t = X_imputed.iloc[train_idx], y.iloc[train_idx]
    
    cat.fit(X_t, y_t)
    xgb.fit(X_t, y_t)
    lgbm.fit(X_t, y_t)
    
    # Accumulate predictions
    test_cat += cat.predict_proba(test_imputed)[:, 1] / (10 * 3)
    test_xgb += xgb.predict_proba(test_imputed)[:, 1] / (10 * 3)
    test_lgbm += lgbm.predict_proba(test_imputed)[:, 1] / (10 * 3)
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import LabelEncoder
from scipy.stats import rankdata

# 1. LOAD DATA [1]
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
test_ids = test['Id']

# 2. FEATURE ENGINEERING [1, 4]
school_freq = train['School'].value_counts().to_dict()

for df in [train, test]:
    # Body Composition & Power
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    df['Power_Index'] = df['Vertical_Jump'] * df['Weight']
    # Scientific Power (Sayers Formula)
    df['Peak_Power_Watts'] = (60.7 * df['Vertical_Jump']) + (45.3 * df['Weight']) - 2055
    
    # Athletic Context
    df['Speed_Score'] = df['Weight'] / df['Sprint_40yd']
    df['Catch_Radius'] = df['Height'] + (df['Vertical_Jump']/100) + (df['Broad_Jump']/100) - (df['Shuttle']/10)
    
    # Ratios (Must handle division by zero/missing) [4]
    df['Agility_Ratio'] = df['Agility_3cone'] / df['Shuttle'].replace(0, np.nan)
    df['Strength_Index'] = df['Bench_Press_Reps'] / df['Weight']
    
    # Position-Relative Features [4]
    for stat in ['Sprint_40yd', 'BMI', 'Catch_Radius']:
        df[f'{stat}_pos_diff'] = df[stat] - df.groupby('Position')[stat].transform('mean')
    
    # Behavioral/Categorical
    df['School_Freq'] = df['School'].map(school_freq).fillna(1)
    df['Is_Young'] = (df['Age'] <= 21).astype(int)
    for col in ['Agility_3cone', 'Shuttle', 'Bench_Press_Reps']:
        df[f'{col}_missing'] = df[col].isnull().astype(int)

# 3. PREPROCESSING [1, 5]
X = train.drop(columns=["Id", "School", "Drafted"])
y = train["Drafted"]
X_test = test[X.columns]

le = LabelEncoder()
for col in ["Player_Type", "Position_Type", "Position"]:
    X[col] = le.fit_transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

# MICE Imputation [5]
imputer = IterativeImputer(random_state=2025)
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=X.columns)

# 4. CRITICAL: NUMERICAL STABILITY [2, 6]
# Replace 'inf' and extreme values that cause XGBoost to crash
for df in [X_imputed, X_test_imputed]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(df.mean(), inplace=True)
    # Clip extreme values to prevent overflow errors
    max_val = 1e6
    df.clip(lower=-max_val, upper=max_val, inplace=True)

# 5. TUNED MODELS [5, 7]
cat = CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05, random_seed=2025, verbose=0)
xgb = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=2025, eval_metric='logloss')
lgbm = LGBMClassifier(n_estimators=250, learning_rate=0.05, random_state=2025, objective='binary')

# 6. ENSEMBLE WITH RANK AVERAGING [3, 8]
# Using 10 splits for higher stability as you reach elite scores
rskf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=42)
test_cat = np.zeros(len(X_test_imputed))
test_xgb = np.zeros(len(X_test_imputed))
test_lgbm = np.zeros(len(X_test_imputed))

for fold, (train_idx, valid_idx) in enumerate(rskf.split(X_imputed, y)):
    X_t, y_t = X_imputed.iloc[train_idx], y.iloc[train_idx]
    
    cat.fit(X_t, y_t)
    xgb.fit(X_t, y_t)
    lgbm.fit(X_t, y_t)
    
    test_cat += cat.predict_proba(X_test_imputed)[:, 1] / (10 * 3)
    test_xgb += xgb.predict_proba(X_test_imputed)[:, 1] / (10 * 3)
    test_lgbm += lgbm.predict_proba(X_test_imputed)[:, 1] / (10 * 3)

# Rank Averaging: Normalizes different model probability distributions [3]
final_preds = (rankdata(test_cat) + rankdata(test_xgb) + rankdata(test_lgbm)) / 3
final_preds /= len(final_preds) # Normalize to 0-1 range

# 7. SUBMISSION [9]
submission = pd.DataFrame({"Id": test_ids, "Drafted": final_preds})
submission.to_csv("submission_target_086.csv", index=False)
print("Rank-Averaged Ensemble created. Stability checks applied.")
# Rank Averaging: Converting probabilities to ranks prevents one model 
# with extreme confidence from drowning out others [1, 2]
rank_cat = rankdata(test_cat) / len(test_cat)
rank_xgb = rankdata(test_xgb) / len(test_xgb)
rank_lgbm = rankdata(test_lgbm) / len(test_lgbm)

final_preds = (0.5 * rank_cat + 0.3 * rank_xgb + 0.2 * rank_lgbm)

# 7. SUBMISSION [2, 13]
submission = pd.DataFrame({"Id": test_ids, "Drafted": final_preds})
submission.to_csv("submission_target_086.csv", index=False)
print("Rank-Averaged Ensemble for 0.86 AUC created successfully.")

c:\PES\GCI\Python\.venv\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


[LightGBM] [Info] Number of positive: 1622, number of negative: 880
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001089 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4563
[LightGBM] [Info] Number of data points in the train set: 2502, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.648281 -> initscore=0.611493
[LightGBM] [Info] Start training from score 0.611493
[LightGBM] [Info] Number of positive: 1622, number of negative: 881
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001232 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4552
[LightGBM] [Info] Number of data points in the train set: 2503, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.648022 -> initscore=0.610358
[LightGBM] [Info] Start training from score 0.610358
[LightGBM] [Info] Number